In [6]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [7]:
# Reading the saved data pickle file
df_texts = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/texts.pkl', compression=None)
df_hourly_sentiment = pd.DataFrame(data=pd.read_csv('/Users/Farid Srouji/Documents/Research/master/globaldata/vix.csv'), 
                        columns=['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23])

df_texts.index = df_texts.Date#["Date"]
df_hourly_sentiment.index = df_hourly_sentiment.Date#["Date"]

df_texts.index= pd.to_datetime(df_texts.index)
df_hourly_sentiment.index= pd.to_datetime(df_hourly_sentiment.index)

In [8]:
df_texts

,Date,Open,High,Low,Close,Adj Close,Volume,0,1,2,...,14,15,16,17,18,19,20,21,22,23
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-25,2019-09-25,17.049999,18.450001,15.690000,15.960000,15.960000,0,,,,...,HILTON EXPANSION BRINGS NEW BRANDS TO CANADA.H...,,,"CARNIVAL CORPORATION, GOVERNMENT OF THE BAHAMA...",,CONSTELLATION BRANDS ELECTS NEW BOARD MEMBERS....,SETS REGULAR QUARTERLY DIVIDEND OF $0.35PER SH...,,ABBOTT ANNOUNCED THE U.S. FOOD AND DRUG ADMINI...,
2019-09-26,2019-09-26,16.230000,17.090000,15.350000,16.070000,16.070000,0,,,,...,,HARVARD UNIVERSITY ESTABLISHES MRNA IMMUNOTHER...,,U.S. JURY RULES IN FAVOR OF EAGLE VIEW TECHNOL...,SETS QUARTERLY DIVIDEND OF $0.20PER SHARE.. UN...,,LYONDELLBASELL PRICES PUBLIC OFFERING OF GUARA...,,ABBVIE ANNOUNCED THAT THE U.S. FOOD AND DRUG A...,
2019-09-27,2019-09-27,15.770000,18.690001,15.150000,17.219999,17.219999,0,,,,...,INTERCONTINENTAL EXCHANGE PROVIDES STATEMENT A...,,BROWN-FORMAN CORPORATION APPOINTS SOPHIA ANGEL...,,,"COOPER COMPANIES INC - ON SEPT 27, CO ENTERED...","ORACLE SAYS IN APRIL, INVESTED $40 MILLION IN ...",,ABBOTT LABORATORIES ANNOUNCED NEW DATA THAT SU...,
2019-09-30,2019-09-30,17.230000,17.350000,16.200001,16.240000,16.240000,0,,,,...,LATE-BREAKING DATA SHOWS ABBOTT'S MITRACLIP™ I...,,KAREN WALKER JOINS INTEL AS SENIOR VICE PRESID...,,"LEGGETT & PLATT INC SAYS ON SEPT 25, MATTHEW ...",AMERICAN TOWER CORPORATION PRICES SENIOR NOTES...,FCB UNITES NORTH AMERICAN OFFICES UNDER NEW ST...,,"PARTECH, INC. ENTERED INTO AGREEMENT TO ACQUIR...",
2019-10-01,2019-10-01,16.020000,18.620001,15.790000,18.559999,18.559999,0,,,,...,JACK DANIEL - THE JACK DANIEL DISTILLERY INTRO...,,XILINX SAYS COLLABORATING WITH MICROSOFT ON AZ...,,DIGITAL REALTY ANNOUNCES PRICING OF PREFERRED ...,WESTERN DIGITAL CORP SAYS CEO STEPHEN MILLIGAN...,,,"ACCENTURE PLC PRESENTS AT ALL DAY DEVOPS, NOV-...",
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-02,2021-08-02,18.160000,19.870001,17.990000,19.459999,19.459999,0,,,,...,,"SYLMAR, CALIF.--(BUSINESS WIRE)--PPG (NYSE:PPG...","NEW YORK, AUG. 2, 2021 /PRNEWSWIRE/ -- WHY: R...","BOSTON, MASSACHUSETTS--(NEWSFILE CORP. - AUGUS...",PHOENIX--(BUSINESS WIRE)--FREEPORT-MCMORAN INC...,"STAMFORD, CONN., AUG. 2, 2021 /PRNEWSWIRE/ -- ...",SECURITIES LITIGATION PARTNER JAMES (JOSH) WIL...,,,
2021-08-03,2021-08-03,19.170000,20.440001,17.700001,18.040001,18.040001,0,,,,...,"NEW YORK, AUG. 3, 2021 /PRNEWSWIRE/ -- ATTORNE...",LOS ANGELES--(BUSINESS WIRE)---- $ATVI #ATVI--...,NEW YORK--(BUSINESS WIRE)--ASSURANT REPORTS SE...,"NEW YORK, AUG. 3, 2021 /PRNEWSWIRE/ -- POMERAN...","NEW YORK, NY / ACCESSWIRE / AUGUST 3, 2021 / T...","NEW YORK, AUG. 3, 2021 /PRNEWSWIRE/ -- ROSEN L...",,"NEW YORK, NY / ACCESSWIRE / AUGUST 3, 2021 / L...",BEIJING--(BUSINESS WIRE)--JUNIPER NETWORKS (NY...,"CHANDLER, ARIZ.--(BUSINESS WIRE)--IN THE RACE ..."
2021-08-04,2021-08-04,18.230000,18.900000,17.670000,17.969999,17.969999,0,,,,...,"RESTON, VA., AUG. 4, 2021 /PRNEWSWIRE/ -- THE ...",CHICAGO--(BUSINESS WIRE)--ADM'S (NYSE: ADM) BO...,"DANVERS, MASS.--(BUSINESS WIRE)--ABIOMED ANNOU...","NEW YORK, AUG. 4, 2021 /PRNEWSWIRE/ -- BERNSTE...","NEW YORK, NEW YORK--(NEWSFILE CORP. - AUGUST 4...",,,NEW YORK--(BUSINESS WIRE)--BRAGAR EAGEL & SQUI...,,


In [9]:
df_hourly_sentiment.fillna(0, inplace=True)

df_hourly_sentiment

,Date,Open,High,Low,Close,Adj Close,Volume,0,1,2,...,14,15,16,17,18,19,20,21,22,23
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-25,2019-09-25,17.049999,18.450001,15.690000,15.960000,15.960000,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2019-09-26,2019-09-26,16.230000,17.090000,15.350000,16.070000,16.070000,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2019-09-27,2019-09-27,15.770000,18.690001,15.150000,17.219999,17.219999,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2019-09-30,2019-09-30,17.230000,17.350000,16.200001,16.240000,16.240000,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2019-10-01,2019-10-01,16.020000,18.620001,15.790000,18.559999,18.559999,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-02,2021-08-02,18.160000,19.870001,17.990000,19.459999,19.459999,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2021-08-03,2021-08-03,19.170000,20.440001,17.700001,18.040001,18.040001,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2021-08-04,2021-08-04,18.230000,18.900000,17.670000,17.969999,17.969999,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
debug=False
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_texts.T.iteritems():
    print("The date is:"+str(date))
    try:
        for hour in range(0, 23):
            sentence = unicodedata.normalize('NFKD', df_texts.loc[date, hour])#.encode('ascii','ignore').decode('ascii')
            # print("The hour is:"+str(hour))
            # print("sentence is")
            # print(sentence)
            ss = sid.polarity_scores(sentence)
            if ((ss['pos'] + ss['neg']) != 0):
                number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
                if (number != ''):
                    df_hourly_sentiment.at[date, hour] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))

            countNeg+=ss['neg']
            countNeu+=ss['neu']
            countPos+=ss['pos']

    except TypeError:
        print("ERROR")
        print("The hour is:"+str(hour))
        print("sentence is")
        print(sentence)
        df_hourly_sentiment.at[date, hour] = 0

        # df.at[date, 'sentiment'] = 0
        # df.at[date, 'sentiment_complementary'] = 0

The date is:2019-09-25 00:00:00
The date is:2019-09-26 00:00:00
The date is:2019-09-27 00:00:00
The date is:2019-09-30 00:00:00
The date is:2019-10-01 00:00:00
The date is:2019-10-02 00:00:00
The date is:2019-10-03 00:00:00
The date is:2019-10-04 00:00:00
The date is:2019-10-07 00:00:00
The date is:2019-10-08 00:00:00
The date is:2019-10-09 00:00:00
The date is:2019-10-10 00:00:00
The date is:2019-10-11 00:00:00
The date is:2019-10-14 00:00:00
The date is:2019-10-15 00:00:00
The date is:2019-10-16 00:00:00
The date is:2019-10-17 00:00:00
The date is:2019-10-18 00:00:00
The date is:2019-10-21 00:00:00
The date is:2019-10-22 00:00:00
The date is:2019-10-23 00:00:00
The date is:2019-10-24 00:00:00
The date is:2019-10-25 00:00:00
The date is:2019-10-28 00:00:00
The date is:2019-10-29 00:00:00
The date is:2019-10-30 00:00:00
The date is:2019-10-31 00:00:00
The date is:2019-11-01 00:00:00
The date is:2019-11-04 00:00:00
The date is:2019-11-05 00:00:00
The date is:2019-11-06 00:00:00
The date

In [11]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df_hourly_sentiment.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/texts_sentiment_hourly.pkl")

Number of Negative words=170.50999999999888
Number of Positive words=733.8979999999989
Number of Neutral words=5269.563000000008


In [13]:
df_hourly_sentiment

,Date,Open,High,Low,Close,Adj Close,Volume,0,1,2,...,14,15,16,17,18,19,20,21,22,23
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-25,2019-09-25,17.049999,18.450001,15.690000,15.960000,15.960000,0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,100.000000,0.000000,100.000000,100.000000,0.000000,84.967320,0.0
2019-09-26,2019-09-26,16.230000,17.090000,15.350000,16.070000,16.070000,0,0.0,0.0,0.0,...,0.000000,100.000000,0.000000,53.381643,100.000000,0.000000,51.724138,0.000000,85.029940,0.0
2019-09-27,2019-09-27,15.770000,18.690001,15.150000,17.219999,17.219999,0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,92.307692,100.000000,0.000000,63.975155,0.0
2019-09-30,2019-09-30,17.230000,17.350000,16.200001,16.240000,16.240000,0,0.0,0.0,0.0,...,100.000000,0.000000,100.000000,0.000000,0.000000,76.335878,63.414634,0.000000,83.673469,0.0
2019-10-01,2019-10-01,16.020000,18.620001,15.790000,18.559999,18.559999,0,0.0,0.0,0.0,...,100.000000,0.000000,0.000000,0.000000,74.125874,0.000000,0.000000,0.000000,75.862069,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-02,2021-08-02,18.160000,19.870001,17.990000,19.459999,19.459999,0,0.0,0.0,0.0,...,0.000000,100.000000,94.017094,68.322981,100.000000,58.333333,46.453901,0.000000,0.000000,0.0
2021-08-03,2021-08-03,19.170000,20.440001,17.700001,18.040001,18.040001,0,0.0,0.0,0.0,...,100.000000,37.864078,86.842105,48.275862,28.977273,51.533742,0.000000,92.125984,100.000000,0.0
2021-08-04,2021-08-04,18.230000,18.900000,17.670000,17.969999,17.969999,0,0.0,0.0,0.0,...,100.000000,100.000000,91.129032,67.283951,82.407407,0.000000,0.000000,60.714286,0.000000,0.0
